## Flow
1) Login with Google OAuth
2) Save login in token.json
3) Connect to Gmail API
4) Fetch latest 10 emails
5) For each email: Get full message + Extract headers
6) Print sender + subject

In [1]:
!pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2

  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import importlib.metadata

# List the distribution package names
packages = ["google-api-python-client", "google-auth",
            "google-auth-oauthlib", "google-auth-httplib2"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


google-api-python-client version: 2.198.0
google-auth version: 2.55.0
google-auth-oauthlib version: 1.4.0
google-auth-httplib2 version: 0.4.0


In [3]:
import os

from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

from googleapiclient.discovery import build

# Read-only access to Gmail
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

creds = None

# Load existing token if present
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file(
        "token.json",
        SCOPES
    )

# If no valid token, authenticate
if not creds or not creds.valid:

    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())

    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json",
            SCOPES
        )

        creds = flow.run_local_server(port=0)

    # Save token
    with open("token.json", "w") as token:
        token.write(creds.to_json())

# Build Gmail service
service = build("gmail", "v1", credentials=creds)

# List the latest 10 emails
results = service.users().messages().list(
    userId="me",
    maxResults=10
).execute()

messages = results.get("messages", [])

print(f"Found {len(messages)} emails.\n")

for msg in messages:

    message = service.users().messages().get(
        userId="me",
        id=msg["id"]
    ).execute()

    headers = message["payload"]["headers"]

    subject = ""
    sender = ""

    for h in headers:
        if h["name"] == "Subject":
            subject = h["value"]

        if h["name"] == "From":
            sender = h["value"]

    print("From :", sender)
    print("Subject :", subject)
    print("-" * 60)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=737381595269-a4d6pt3407tq0okofigrvjjntlueaspc.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49189%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=1NDttpc9zMhsEnNgqO7n1FVlK2wF8o&code_challenge=reP8ph3SDsXRY8r8ZeUoqb8I2VEx6GWxd2hdc0tb9ZU&code_challenge_method=S256&access_type=offline
Found 10 emails.

From : Glassdoor Jobs <noreply@glassdoor.com>
Subject : New jobs in Delhi, India. Apply Now.
------------------------------------------------------------
From : edX <edX@news.edx.org>
Subject : You've almost reached your goal in CS50's Introduction to Artificial Intelligence with Python
------------------------------------------------------------
From : Rizernet Solutions <dhana.scml6ZXJuZXQuY29t@naukri.com>
Subject : ✉️ Job | AI Native engineer in Hyderabad, Bengaluru, Indore
--------------------------------------------------